# Engenharia de Dados - Projeto E-Commerce (Camada bronze)
**Objetivo:** Realizar a extração e carga dos dados brutos (CSV e API) para o Data Lakehouse. Os dados são armazenados em tabelas preservando sua estrutura original, acrescidos de um timestamp de ingestão para fins de auditoria e controle.

## Estrutura e configurações iniciais
**Objetivo:** Definir o catálogo e o schema bronze

In [0]:
# Definindo o catálogo de trabalho
spark.sql("USE CATALOG projeto_medalhao_visagio")

# Garantindo a existência do schema Silver para persistência das tabelas transformadas
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

# Definindo o path do volume
path_volume = "/Volumes/projeto_medalhao_visagio/raw_zone/landing_olist/"

## Função de Ingestão de CSVs
**Objetivo:** Padronizar a leitura de arquivos CSV a partir dos Volumes

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

def ingest_csv_to_bronze(file_name, table_name):
      try:
            df = (spark.read
                  .format("csv")
                  .option("header", "true")
                  .option("sep", ",")
                  .option("mode", "PERMISSIVE")
                  .load(f"{path_volume}{file_name}"))
            
            # Adicionando o timestamp de ingestão
            df = df.withColumn("timestamp_ingestion", F.current_timestamp())

            # Gravando no Lakehouse
            (df.write
                  .format("delta")
                  .mode("overwrite")
                  .option("overwriteSchema", "true")
                  .saveAsTable(f"bronze.{table_name}"))
            
            print(f"Tabela bronze.{table_name} carregada com sucesso.")
            
      except Exception as e:
            print(f"Erro ao carregar tabela bronze.{table_name}: {str(e)}")

## Execução do Mapeamento
**Objetivo:** Garantir que cada arquivo seja convertido em sua respectiva tabela na camada Bronze

In [0]:
mapeamento = {
    "olist_customers_dataset.csv": "tb_customers",
    "olist_geolocation_dataset.csv": "tb_geolocalizacao",
    "olist_order_items_dataset.csv": "tb_order_items",
    "olist_order_payments_dataset.csv": "tb_order_payments",
    "olist_order_reviews_dataset.csv": "tb_order_reviews",
    "olist_orders_dataset.csv": "tb_orders",
    "olist_products_dataset.csv": "tb_products",
    "olist_sellers_dataset.csv": "tb_sellers",
    "product_category_name_translation.csv": "tb_product_category_name_translation"
}

for csv, tabela in mapeamento.items():
    ingest_csv_to_bronze(csv, tabela)

Tabela bronze.tb_customers carregada com sucesso.
Tabela bronze.tb_geolocalizacao carregada com sucesso.
Tabela bronze.tb_order_items carregada com sucesso.
Tabela bronze.tb_order_payments carregada com sucesso.
Tabela bronze.tb_order_reviews carregada com sucesso.
Tabela bronze.tb_orders carregada com sucesso.
Tabela bronze.tb_products carregada com sucesso.
Tabela bronze.tb_sellers carregada com sucesso.
Tabela bronze.tb_product_category_name_translation carregada com sucesso.


## Configuração de Parâmetros e Janela Temporal
**Objetivo:** Define o intervalo de datas para a extração da cotação via API

In [0]:
# Definir os widgets
dbutils.widgets.text("data_inicio", "")
dbutils.widgets.text("data_fim", "")

# Capturar os valores dos widgets
dt_ini_param = dbutils.widgets.get("data_inicio")
dt_fim_param = dbutils.widgets.get("data_fim")

# Lógica de decisão: Se o widget estiver vazio, busca no banco (Automação)
if dt_ini_param == "" or dt_fim_param == "":
    datas_pedidos = spark.table("bronze.tb_orders").select(
        F.min(F.to_timestamp("order_purchase_timestamp")).alias("min_data"),
        F.max(F.to_timestamp("order_purchase_timestamp")).alias("max_data")
    ).collect()
    
    dt_ini = datas_pedidos[0]["min_data"].strftime("%m-%d-%Y")
    dt_fim = datas_pedidos[0]["max_data"].strftime("%m-%d-%Y")
    print(f"Widgets vazios. Usando datas automáticas: {dt_ini} até {dt_fim}")
else:
    dt_ini = dt_ini_param
    dt_fim = dt_fim_param
    print(f"Usando datas fornecidas via parâmetros: {dt_ini} até {dt_fim}")

Widgets vazios. Usando datas automáticas: 09-04-2016 até 10-17-2018


## Ingestão da API (Cotação do Dólar)
**Objetivo:** Realizar a extração de dados financeiros diretamente da API do Banco Central utilizando os parâmetros de data configurados.

In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F
from datetime import datetime

# Endpoint do Banco Central
url_api = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{dt_ini}'&@dataFinalCotacao='{dt_fim}'&"
    f"$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

try:
    response = requests.get(url_api)
    if response.status_code == 200:
        dados_api = response.json()['value']

        df_api = spark.createDataFrame(pd.DataFrame(dados_api)) 
        df_api = df_api.withColumn("timestamp_ingestion", F.current_timestamp())
        
        # Salvar no Lakehouse
        (df_api.write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable("bronze.tb_cotacao_dolar"))
        
        print("Tabela bronze.tb_cotacao_dolar atualizada.")
    else:
        print(f"Erro na API: Status {response.status_code}")
except Exception as e:
    print(f"Falha na conexão com a API: {e}")

Tabela bronze.tb_cotacao_dolar atualizada.
